<a href="https://colab.research.google.com/github/NoCode-NoLife666/Anime-Recommendation/blob/main/Anime_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install stuff
!pip install xgboost optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 22.3 MB/s eta 0:00:00


In [ ]:
#Import stuff

import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr
import xgboost as xgb
import optuna
import kagglehub
import os



In [ ]:
dataset_dir = kagglehub.dataset_download("syahrulapriansyah2/myanimelist-2025")

100%|██████████| 9.57M/9.57M [00:00<00:00, 20.0MB/s]

Extracting files...


In [ ]:
#Config. Change the file names acc to what you have

ANIMELIST_XML     = "animelistv1.xml"   # your exported MAL XML filename
ANIME_CSV = os.path.join(dataset_dir, "mal_anime.csv")
# MAL anime dataset from kaggle
N_CV_FOLDS        = 5     # folds used inside the 80% train split for hyperparam search
OPTUNA_TRIALS     = 20
EARLY_STOP_ROUNDS = 20
MAX_ESTIMATORS    = 500   # ceiling; early stopping decides the real number

In [ ]:
#Constants

ALL_GENRES = [
    'Action', 'Adventure', 'Avant Garde', 'Award Winning', 'Cars', 'Comedy',
    'Demons', 'Drama', 'Ecchi', 'Fantasy', 'Game', 'Gourmet', 'Harem', 'Hentai',
    'Historical', 'Horror', 'Josei', 'Martial Arts', 'Mecha', 'Military', 'Music',
    'Mystery', 'Parody', 'Police', 'Psychological', 'Romance', 'Samurai', 'School',
    'Sci-Fi', 'Seinen', 'Shoujo', 'Shounen', 'Slice of Life', 'Space', 'Sports',
    'Super Power', 'Supernatural', 'Suspense', 'Vampire', 'Work Life'
]
ALL_TYPES   = ['type_Movie', 'type_ONA', 'type_OVA', 'type_Special', 'type_TV']
ALL_SOURCES = [
    'src_4-koma manga', 'src_Game', 'src_Light novel', 'src_Manga',
    'src_Mixed media', 'src_Music', 'src_Novel', 'src_Original',
    'src_Other', 'src_Visual novel', 'src_Web manga'
]
STOPWORDS = {'on', 'the', 'a', 'an', 'in', 'of', 'to', 'and', 'no',
             'wa', 'ga', 'wo', 'ni', 'de', 'mo', 'ka', 'ha'}
GENRE_ALIASES = {
    'Erotica': 'Hentai',  # mal_anime.csv splits these; fold Erotica into Hentai
}


In [ ]:
#Helper functions

def encode_genres(genre_str):
    """Comma-separated genre string → multi-hot list."""
    genres = set(g.strip() for g in genre_str.split(',')) if pd.notna(genre_str) and genre_str else set()
    genres = {GENRE_ALIASES.get(g, g) for g in genres}
    return [1 if g in genres else 0 for g in ALL_GENRES]


def build_feature_row(row, num_eps_median, score_mean, year_median):
    """Build a single-row feature DataFrame from an anime_df row."""
    genre_vals = encode_genres(row.get('genres', ''))
    genre_cols = [f'genre_{g.lower().replace(" ", "_")}' for g in ALL_GENRES]
    feat = dict(zip(genre_cols, genre_vals))

    for col in ALL_TYPES:
        feat[col] = 1 if col == f'type_{row.get("type", "")}' else 0
    for col in ALL_SOURCES:
        feat[col] = 1 if col == f'src_{row.get("source_type", "")}' else 0

    feat['num_episodes']    = row['num_episodes']    if pd.notna(row.get('num_episodes'))  else num_eps_median
    feat['mal_score']       = row['score']           if pd.notna(row.get('score'))         else score_mean
    feat['popularity_rank'] = row.get('popularity_rank', 99999)
    feat['favorites_count'] = row.get('favorites_count', 0)
    feat['members_count']   = row.get('members_count', 0)
    feat['start_year']      = row['start_year']      if pd.notna(row.get('start_year'))    else year_median

    return pd.DataFrame([feat])


def search_anime(query: str, df: pd.DataFrame, top_n: int = 5):
    """
    Search anime dataset by title.
    Uses substring match first, then word-overlap fallback.
    Punctuation-insensitive (handles Steins;Gate → Steins Gate etc).
    """
    import re
    def normalise(s):
        return re.sub(r"[^\w\s]", " ", str(s).lower()).strip()

    q      = normalise(query)
    q_words = set(q.split()) - STOPWORDS

    norm_titles = df['title'].apply(normalise)

    mask = norm_titles.str.contains(q, regex=False, na=False)
    sub  = df[mask].copy()

    def overlap(norm_title):
        t_words = set(norm_title.split()) - STOPWORDS
        if not q_words or not t_words:
            return 0
        inter    = q_words & t_words
        jaccard  = len(inter) / len(q_words | t_words)
        coverage = len(inter) / len(q_words)
        return 0.4 * jaccard + 0.6 * coverage

    df2        = df.copy()
    df2['_s']  = norm_titles.apply(overlap)
    word       = df2[df2['_s'] >= 0.5].sort_values(['_s', 'popularity_rank'], ascending=[False, True])

    combined = pd.concat([sub, word]).drop_duplicates('anime_id')
    return combined.sort_values('popularity_rank').head(top_n)


def compute_metrics(y_true, y_pred):
    """
    Full error report, not just MAE/RMSE:
      - mae      : avg absolute points off (interpretable on the 1-10 scale)
      - rmse     : same units as MAE, penalises big misses harder
      - r2       : how much of the variance in your scores the model explains
      - spearman : rank correlation - matters most for rank_ptw(), since
                   getting the ORDER right matters more than the exact number
      - mape     : average error as a % of the true score
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae      = mean_absolute_error(y_true, y_pred)
    rmse     = np.sqrt(mean_squared_error(y_true, y_pred))
    r2       = r2_score(y_true, y_pred)
    spearman = spearmanr(y_true, y_pred).correlation
    mape     = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'spearman': spearman, 'mape': mape}



In [ ]:
#load the my anime list

def load_xml(path):
    tree = ET.parse(path)
    root = tree.getroot()
    rows = [{c.tag: c.text for c in a} for a in root.findall('anime')]
    df   = pd.DataFrame(rows)
    df['series_animedb_id'] = df['series_animedb_id'].astype(int)
    df['my_score']          = df['my_score'].astype(int)
    return df

xml_df    = load_xml(ANIMELIST_XML)
train_xml = xml_df[(xml_df['my_status'] == 'Completed') & (xml_df['my_score'] > 0)].copy()
print(f"Completed + scored entries : {len(train_xml)}")


Completed + scored entries : 730


In [ ]:
#load mai_anime.csv



def _clean_numeric(series):
    return pd.to_numeric(
        series.astype(str).str.replace('#', '', regex=False).str.replace(',', '', regex=False),
        errors='coerce'
    )

try:
    _raw = pd.read_csv(ANIME_CSV, sep=",")
except pd.errors.ParserError as e:

    # Some rows have edge-case quote escaping (e.g. nested quotes inside
    # synopsis text) that trips up the fast C parser on certain pandas
    # versions. The Python engine is slower but far more tolerant of this.
    print(f"⚠️  C parser failed ({e}); retrying with the Python engine...")
    _raw = pd.read_csv(ANIME_CSV, sep=",", engine="python", on_bad_lines="warn")
print(f"Loaded {len(_raw)} raw rows from {ANIME_CSV}")

_ALL_GENRES_SET = set(ALL_GENRES)

#Genres/Demographic/Themes are three separate columns in mal dataset. added them together into one genre column.

def _merge_genre_sources(row):
    tags = set()
    for col in ('Genres', 'Demographic', 'Themes'):
        val = row.get(col)
        if pd.notna(val):
            for t in str(val).split(','):
                t = GENRE_ALIASES.get(t.strip(), t.strip())
                if t in _ALL_GENRES_SET:
                    tags.add(t)
    return ', '.join(sorted(tags))

_merged_genres = _raw.apply(_merge_genre_sources, axis=1)

anime_df = pd.DataFrame({
    'anime_id':        _raw['myanimelist_id'],
    'title':           _raw['title'],
    'type':            _raw['Type'].replace({'TV Special': 'Special'}),  # fold into existing category
    'source_type':     _raw['Source'],
    'genres':          _merged_genres,
    'score':           _raw['Score'],
    'num_episodes':    _clean_numeric(_raw['Episodes']),
    'popularity_rank': _clean_numeric(_raw['Popularity']),
    'favorites_count': _clean_numeric(_raw['Favorites']),
    'members_count':   _clean_numeric(_raw['Members']),
    'start_year':      _raw['Released_Year'],
})
print(f"Anime dataset               : {len(anime_df)} titles")


Loaded 19931 raw rows from /root/.cache/kagglehub/datasets/syahrulapriansyah2/myanimelist-2025/versions/1/mal_anime.csv
Anime dataset               : 19931 titles


In [ ]:
merged = train_xml.merge(anime_df, left_on='series_animedb_id', right_on='anime_id', how='inner')
print(f"Matched in dataset          : {len(merged)}")
print(f"Missing (too new / no match): {len(train_xml) - len(merged)}")

Matched in dataset          : 728
Missing (too new / no match): 2


In [ ]:
NUM_EPS_MEDIAN = merged['num_episodes'].median()
SCORE_MEAN     = merged['score'].mean()
YEAR_MEDIAN    = merged['start_year'].median()


GENRE_MIN_COUNT = 3 #drops anime seens less than this amount. useful to avoid overfitting.

_genre_counts = pd.Series(0, index=ALL_GENRES)
for _g_str in merged['genres'].dropna():
    for _g in {GENRE_ALIASES.get(x.strip(), x.strip()) for x in _g_str.split(',')}:
        if _g in _genre_counts.index:
            _genre_counts[_g] += 1

_dropped_genres = _genre_counts[_genre_counts < GENRE_MIN_COUNT].index.tolist()
ALL_GENRES = [g for g in ALL_GENRES if g not in _dropped_genres]
print(f"Dropping {len(_dropped_genres)} sparse genres (<{GENRE_MIN_COUNT} watched titles): {_dropped_genres}")
print(f"Keeping {len(ALL_GENRES)} genres")

genre_matrix   = pd.DataFrame(
    merged['genres'].apply(encode_genres).tolist(),
    columns=[f'genre_{g.lower().replace(" ", "_")}' for g in ALL_GENRES]
)
type_dummies   = pd.get_dummies(merged['type'],        prefix='type').reindex(columns=ALL_TYPES,   fill_value=0)
source_dummies = pd.get_dummies(merged['source_type'], prefix='src' ).reindex(columns=ALL_SOURCES, fill_value=0)
numeric        = pd.DataFrame({
    'num_episodes':    merged['num_episodes'].fillna(NUM_EPS_MEDIAN).values,
    'mal_score':       merged['score'].fillna(SCORE_MEAN).values,
    'popularity_rank': merged['popularity_rank'].values,
    'favorites_count': merged['favorites_count'].values,
    'members_count':   merged['members_count'].values,
    'start_year':      merged['start_year'].fillna(YEAR_MEDIAN).values,
})

X = pd.concat([
    genre_matrix.reset_index(drop=True),
    type_dummies.reset_index(drop=True),
    source_dummies.reset_index(drop=True),
    numeric.reset_index(drop=True)
], axis=1)
y = merged['my_score'].reset_index(drop=True)

FEATURE_COLUMNS = X.columns.tolist()
print(f"Feature matrix               : {X.shape[0]} rows × {X.shape[1]} features")


Dropping 6 sparse genres (<3 watched titles): ['Cars', 'Demons', 'Game', 'Police', 'Space', 'Work Life']
Keeping 34 genres
Feature matrix               : 728 rows × 56 features


In [ ]:
#training set and validation set split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {len(X_train)}  |  Val: {len(X_val)}")

Train: 582  |  Val: 146


In [ ]:
# hyperparam search using optuna

print(f"\n── Running Optuna ({OPTUNA_TRIALS} trials) ──")
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    param = {
        'n_estimators': MAX_ESTIMATORS,
        'max_depth': trial.suggest_int('max_depth', 2, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 10.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
        'objective': 'reg:squarederror',
        'early_stopping_rounds': EARLY_STOP_ROUNDS,
        'random_state': 42,
        'verbosity': 0
    }

    kf = KFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=42)
    rmse_scores = []

    for tr_idx, va_idx in kf.split(X_train):
        X_tr, y_tr = X_train.iloc[tr_idx], y_train.iloc[tr_idx]
        X_va, y_va = X_train.iloc[va_idx], y_train.iloc[va_idx]

        model = xgb.XGBRegressor(**param)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        preds = np.clip(model.predict(X_va), 1, 10)
        rmse_scores.append(np.sqrt(mean_squared_error(y_va, preds)))

    return np.mean(rmse_scores)

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

best_params = study.best_params
best_params['n_estimators'] = MAX_ESTIMATORS
best_params['objective'] = 'reg:squarederror'
best_params['early_stopping_rounds'] = EARLY_STOP_ROUNDS
best_params['random_state'] = 42
best_params['verbosity'] = 0

print("✅ Best hyperparameters found:")
for key, value in best_params.items():
    if key not in ['random_state', 'verbosity', 'n_estimators', 'objective', 'early_stopping_rounds']:
        print(f"   {key}: {value}")




── Running Optuna (20 trials) ──


  0%|          | 0/20 [00:00<?, ?it/s]

✅ Best hyperparameters found:
   max_depth: 3
   learning_rate: 0.17254716573280354
   subsample: 0.7927975767245621
   colsample_bytree: 0.7394633936788146
   reg_alpha: 0.7800932022121826
   reg_lambda: 2.403950683025824
   min_child_weight: 1


In [ ]:
#training the model
X_train_fit, X_es, y_train_fit, y_es = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42
)

model = xgb.XGBRegressor(**best_params)
model.fit(X_train_fit, y_train_fit, eval_set=[(X_es, y_es)], verbose=False)
print(f"\n✅ Final model trained. Stopped at {model.best_iteration} trees "
      f"(ceiling was {MAX_ESTIMATORS}).")



✅ Final model trained. Stopped at 13 trees (ceiling was 500).


In [ ]:
#error metrics

HR = "-" * 60

train_preds = np.clip(model.predict(X_train_fit), 1, 10)
val_preds   = np.clip(model.predict(X_val), 1, 10)

train_metrics = compute_metrics(y_train_fit.values, train_preds)
val_metrics   = compute_metrics(y_val.values, val_preds)

print(HR)
print("TRAINING vs VALIDATION")
print(HR)
print(f"{'Metric':<10}{'Train':>10}{'Val':>10}{'Gap':>10}")
for label, key, fmt in [
    ('MAE',      'mae',      '.4f'),
    ('RMSE',     'rmse',     '.4f'),
    ('R2',       'r2',       '.4f'),
    ('Spearman', 'spearman', '.4f'),
    ('MAPE %',   'mape',     '.2f'),
]:
    tr, va = train_metrics[key], val_metrics[key]
    print(f"{label:<10}{tr:>10{fmt}}{va:>10{fmt}}{va-tr:>10{fmt}}")

print(f"\nOn average, validation predictions are off by ±{val_metrics['mae']:.2f} pts "
      f"(~{val_metrics['mape']:.1f}%) on a 1-10 scale.")
print(f"R2 of {val_metrics['r2']:.3f} on validation - model explains that fraction of the "
      f"variance in your actual scores.")
print(f"Spearman rank correlation of {val_metrics['spearman']:.3f} on validation - this is what "
      f"matters most for rank_ptw(), since getting the ORDER of your PTW list right is the goal, "
      f"not the exact number.")

print(f"\n{HR}")
print("ACTUAL vs AVG PREDICTED (validation set)")
print(HR)
comp = pd.DataFrame({'actual': y_val.values, 'predicted': np.round(val_preds).astype(int)})
print(comp.groupby('actual')['predicted'].agg(['mean', 'count'])
      .rename(columns={'mean': 'avg_predicted', 'count': 'n_anime'}).round(2))

print(f"\n{HR}")
print("TOP 15 FEATURES BY IMPORTANCE")
print(HR)
feat_imp = pd.Series(model.feature_importances_, index=FEATURE_COLUMNS).sort_values(ascending=False)
print(feat_imp.head(15).round(4).to_string())


------------------------------------------------------------
TRAINING vs VALIDATION
------------------------------------------------------------
Metric         Train       Val       Gap
MAE           0.6606    0.7188    0.0582
RMSE          0.8231    0.9228    0.0997
R2            0.5850    0.4176   -0.1674
Spearman      0.7737    0.6660   -0.1077
MAPE %          9.18     10.04      0.86

On average, validation predictions are off by ±0.72 pts (~10.0%) on a 1-10 scale.
R2 of 0.418 on validation - model explains that fraction of the variance in your actual scores.
Spearman rank correlation of 0.666 on validation - this is what matters most for rank_ptw(), since getting the ORDER of your PTW list right is the goal, not the exact number.

------------------------------------------------------------
ACTUAL vs AVG PREDICTED (validation set)
------------------------------------------------------------
        avg_predicted  n_anime
actual                        
4                7.00        

In [ ]:
#predict function (need to rank the PTW list)

WATCHED_IDS = set(xml_df['series_animedb_id'].tolist())

def predict(query: str, top_n: int = 3):
    """
    Predict your score for any anime in the dataset.

    Usage:
        predict("Fullmetal Alchemist Brotherhood")
        predict("One Punch Man")
        predict("Steins Gate")          # semicolons optional
        predict("Shingeki no Kyojin")   # JP title for Attack on Titan
        predict("Kimetsu no Yaiba")     # JP title for Demon Slayer
        predict("Shigatsu wa Kimi no Uso")  # Your Lie in April
    """
    print(f"\nSearch: '{query}'")
    print(HR)

    results = search_anime(query, anime_df, top_n=top_n)

    if results.empty:
        print("Not found in dataset.")
        print("Tip: try the Japanese title (e.g. 'Shingeki no Kyojin')")
        return

    for i, (_, row) in enumerate(results.iterrows()):
        feat  = build_feature_row(row, NUM_EPS_MEDIAN, SCORE_MEAN, YEAR_MEDIAN)
        feat  = feat.reindex(columns=FEATURE_COLUMNS, fill_value=0)
        score = float(np.clip(model.predict(feat)[0], 1, 10))
        bar   = "#" * round(score) + "." * (10 - round(score))
        watched = int(row['anime_id']) in WATCHED_IDS

        print(f"\n[{i+1}] {row['title']}")
        print(f"    Type: {row['type']}   Source: {row['source_type']}   Year: {row.get('start_year', '?')}")
        print(f"    Episodes: {row['num_episodes']}   MAL Score: {row['score']}")
        print(f"    Genres: {row['genres']}")
        print(f"    ⭐ Predicted score: {score:.1f}/10   [{bar}]")

        if watched:
            actual = xml_df.loc[xml_df['series_animedb_id'] == row['anime_id'], 'my_score'].values
            if len(actual) and actual[0] > 0:
                diff = abs(score - actual[0])
                print(f"    ✅ Already watched - your score: {actual[0]}/10  (model off by {diff:.1f})")
            else:
                print(f"    In your list (PTW or unscored)")

    print(f"\n{HR}")
    if len(results) > 1:
        print("Results sorted by MAL popularity. Top result is usually correct.")


In [ ]:
#to predict one by one as explained in the docstring above.:
predict("Monogatari")


Search: 'Monogatari'
------------------------------------------------------------

[1] Bakemonogatari
    Type: TV   Source: Light novel   Year: 2009.0
    Episodes: 15.0   MAL Score: 8.32
    Genres: Mystery, Romance, Supernatural, Vampire
    ⭐ Predicted score: 8.3/10   [########..]
    ✅ Already watched - your score: 8/10  (model off by 0.3)

[2] Nisemonogatari
    Type: TV   Source: Light novel   Year: 2012.0
    Episodes: 11.0   MAL Score: 8.12
    Genres: Comedy, Ecchi, Mystery, Supernatural
    ⭐ Predicted score: 8.0/10   [########..]

[3] Ore Monogatari!!
    Type: TV   Source: Manga   Year: 2015.0
    Episodes: 24.0   MAL Score: 7.89
    Genres: Comedy, Romance, Shoujo
    ⭐ Predicted score: 8.1/10   [########..]
    ✅ Already watched - your score: 8/10  (model off by 0.1)

------------------------------------------------------------
Results sorted by MAL popularity. Top result is usually correct.


In [ ]:
#ranking the PTW shows

def rank_ptw():
    ptw        = xml_df[xml_df['my_status'] == 'Plan to Watch'].copy()
    ptw_merged = ptw.merge(anime_df, left_on='series_animedb_id', right_on='anime_id', how='inner')
    missing    = ptw[~ptw['series_animedb_id'].isin(ptw_merged['anime_id'])]

    print(f"PTW total: {len(ptw)}  |  In dataset: {len(ptw_merged)}  |  Missing (too new): {len(missing)}")

    results = []
    for _, row in ptw_merged.iterrows():
        feat  = build_feature_row(row, NUM_EPS_MEDIAN, SCORE_MEAN, YEAR_MEDIAN)
        feat  = feat.reindex(columns=FEATURE_COLUMNS, fill_value=0)
        score = float(np.clip(model.predict(feat)[0], 1, 10))
        results.append({
            'title':     row['title'],
            'score':     round(score, 2),
            'type':      row['type'],
            'episodes':  row['num_episodes'],
            'mal_score': row['score'],
        })

    ranked = pd.DataFrame(results).sort_values('score', ascending=False).reset_index(drop=True)
    ranked.index += 1

    print(f"\n{HR}")
    print("YOUR PTW LIST - RANKED BY PREDICTED SCORE")
    print(HR)
    print(f"  {'#':<4} {'Title':<38} {'Pred':>5}  {'MAL':>5}  {'Eps':>4}  Type")
    print(HR)
    for rank, row in ranked.iterrows():
        mal_str = f"{row['mal_score']:.2f}" if pd.notna(row['mal_score']) else "  N/A"
        eps_str = str(int(row['episodes'])) if pd.notna(row['episodes']) else "?"
        print(f"  {rank:<4} {row['title'][:37]:<38} {row['score']:>5.1f}  {mal_str:>5}  {eps_str:>4}  {row['type']}")
    print(HR)

    if len(missing):
        print(f"\nNot in dataset (too new to predict):")
        for _, row in missing.iterrows():
            print(f"  - {row['series_title']}")

    return ranked

ptw_rankings = rank_ptw()


PTW total: 133  |  In dataset: 125  |  Missing (too new): 8

------------------------------------------------------------
YOUR PTW LIST - RANKED BY PREDICTED SCORE
------------------------------------------------------------
  #    Title                                   Pred    MAL   Eps  Type
------------------------------------------------------------
  1    Cyberpunk: Edgerunners                   9.0   8.62    10  ONA
  2    Gintama                                  9.0   8.93   201  TV
  3    Ginga Eiyuu Densetsu                     8.9   9.02   110  OVA
  4    Banana Fish                              8.8   8.45    24  TV
  5    Psycho-Pass                              8.8   8.33    22  TV
  6    Nichijou                                 8.7   8.47    26  TV
  7    Ookami Kodomo no Ame to Yuki             8.5   8.56     1  Movie
  8    Mahou Shoujo Madoka★Magica               8.5   8.38    12  TV
  9    Shiguang Dailiren II                     8.5   8.64    12  ONA
  10   Pluto    